# 03 — Accounts and coordination

Two separate questions that the dashboard is at risk of conflating:

- **Is this account automated?** (`bot_prob`, a supervised model)
- **Is this account acting in concert with others?** (`coordination_score`,
  a transparent formula over a graph)

They are independent. A coordinated campaign can be run entirely by humans,
and a bot can be a weather feed nobody coordinates with. Neither score is
evidence for the other, and this notebook keeps them in separate sections
on purpose.

**The finding in this notebook is the null-model comparison**, not the
communities. Any graph has communities; Louvain will happily partition
random noise and report a modularity above zero.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modeling.config import get_settings, run_fingerprint, set_all_seeds
from modeling.io import CorpusReader, ScoredStore

set_all_seeds()
settings = get_settings()
reader = CorpusReader(settings)
store = ScoredStore(settings)

# Every number below is tied to this fingerprint. If a rerun disagrees with
# a committed figure, this block is where the diagnosis starts.
fingerprint = run_fingerprint()
print(f"seed={fingerprint['seed']}  device={fingerprint['device']}  "
      f"corpus={fingerprint['input_manifest_hash']}")

## Feature tiers, and why the bot model may be null

Follower and following counts exist for Mastodon and for the Twitter
benchmarks. They do not exist for ConvoKit Reddit at all. A model trained on
TwiBot-22's forty Twitter features and scored on the twelve this corpus can
compute is not a model — it is a lookup table for a platform we do not have.

`modeling/accounts/features.py` declares three tiers and the classifier
trains on the intersection of what the benchmark and the corpus can both
supply. When the intersection is empty, `bot_prob` is **null with a reason
code** rather than a number.

In [ ]:
from modeling.accounts.features import available_tiers, build_features

records = reader.records()
authors = reader.authors()
tiers = available_tiers(records, authors)
print('tiers this corpus supports:', tiers)

features = build_features(records, authors, tiers=tiers)
frame = features.as_frame()
print(f'{frame.shape[0]} accounts x {frame.shape[1]} features')
frame.describe().T[['mean', 'std', 'min', '50%', 'max']].round(3)

In [ ]:
cols = ['hour_entropy', 'burstiness', 'duplicate_content_rate', 'self_similarity_mean']
present = [c for c in cols if c in frame.columns]
frame[present].hist(bins=30, figsize=(11, 6))
plt.suptitle('behavioural features — look for bimodality, not outliers')
plt.tight_layout()

## Account scores

`skip_reasons` is the column to read first. It says why a null is null, and
a table of nulls with no reasons would be unexplainable three weeks later.

In [ ]:
author_scores = store.read('author_scores')
if not len(author_scores):
    print('Nothing scored yet. Run: python -m modeling.cli score --all')
else:
    print(f'{len(author_scores)} authors')
    for column in ['bot_prob', 'coordination_score', 'community_id', 'toxicity_mean']:
        if column in author_scores:
            filled = author_scores[column].notna().sum()
            print(f'  {column:22} {filled:5}/{len(author_scores)} populated')
    from collections import Counter
    reasons = Counter(r for rs in author_scores['skip_reasons'].dropna() for r in rs)
    print('\nskip reasons:', dict(reasons))

## The coordination graph

Edges carry an **evidence type**, so the UI can say *why* two accounts are
linked rather than merely asserting that they are:

| evidence | meaning |
|---|---|
| `near_dup` | near-identical content inside the window |
| `cotweet` | the same URL or domain inside the window |
| `hashtag_seq` | the same *ordered* hashtag sequence |
| `temporal` | replies to the same parent inside a tight window |

Ordering matters for `hashtag_seq`: a shared ordering is much stronger
evidence of a shared template than a shared vocabulary.

In [ ]:
edges = store.read('coordination_edges')
if not len(edges):
    print('No edges. Run: python -m modeling.cli score coordination')
else:
    print(f'{len(edges)} edges')
    display(edges['evidence'].value_counts())
    edges['weight'].plot.hist(bins=30, figsize=(6, 3))
    plt.title('edge weight distribution'); plt.tight_layout()

## The null model — this is the finding

Timestamps are shuffled **within each author** and the whole pipeline re-run.
That destroys cross-account timing coincidences — the thing coordination
detection claims to find — while preserving every author's own volume,
burstiness and diurnal rhythm. A *global* shuffle would destroy those too,
and would be too easy a null to beat.

If observed modularity does not exceed the shuffled mean by at least one
standard deviation, **'we found coordinated communities' is not a result**,
and the cell below says so in as many words.

In [ ]:
from modeling.accounts.coordination import CoordinationDetector, null_model_section

result = CoordinationDetector(settings).detect(records)
print(null_model_section(result))
print()
print(result.summary())

In [ ]:
if result.edges:
    import networkx as nx
    graph = nx.Graph()
    for edge in result.edges:
        graph.add_edge(edge['src_author_id'], edge['dst_author_id'],
                       weight=edge['weight'])
    largest = max(nx.connected_components(graph), key=len)
    subgraph = graph.subgraph(list(largest)[:60])
    plt.figure(figsize=(7, 6))
    nx.draw_networkx(subgraph, pos=nx.spring_layout(subgraph, seed=settings.seed),
                     node_size=90, font_size=6, width=0.6, with_labels=False)
    plt.title(f'largest component ({len(largest)} accounts)'); plt.axis('off')